# 02 似た動きをさがそう:相関と因果

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cogproject/stats-ml-bio-study/blob/main/notebooks/02_correlation.ipynb)

**今日のゴール(30分)**
1. 2つの遺伝子が「いっしょに増えたり減ったりする」かを、グラフと数字で確かめる(**相関**)
2. 「いっしょに動く」=「片方がもう片方の原因」ではない、ということを、本物のデータで体験する(**相関と因果のちがい**)

データは 01 と同じ、がんの患者さん801人 × 遺伝子2万個の表です(くわしくは 01 を見てください)。

## 0. 準備(01 と同じ。まとめて実行してください、1〜2分)

In [ ]:
!pip -q install japanize-matplotlib
import japanize_matplotlib
import urllib.request, zipfile, tarfile, glob, os
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

os.makedirs("data", exist_ok=True)
url = "https://archive.ics.uci.edu/static/public/401/gene+expression+cancer+rna+seq.zip"
zip_path = "data/rnaseq.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("data")
for tgz in glob.glob("data/**/*.tar.gz", recursive=True):
    with tarfile.open(tgz) as t:
        t.extractall("data")
X = pd.read_csv(glob.glob("data/**/data.csv", recursive=True)[0], index_col=0)
y = pd.read_csv(glob.glob("data/**/labels.csv", recursive=True)[0], index_col=0)["Class"]

# 01 と同じく、人によって値がよく変わる遺伝子を500個にしぼる
X_top = X[X.var().sort_values(ascending=False).index[:500]]
print("準備できました:", X_top.shape)

## 1. 相関ってなに?
**相関(そうかん)** は、「片方が大きいと、もう片方も大きい(または小さい)」という関係のことです。

- 身長と体重:背が高い人ほど体重も重いことが多い → **正の相関**(右上がり)
- 気温とホットコーヒーの売れ行き:暑いほど売れない → **負の相関**(右下がり)
- 身長とテストの点数:ほぼ関係なし → **相関なし**

この強さを −1 から +1 の数字で表したものを **相関係数** といいます。
+1 に近いほど右上がり、−1 に近いほど右下がり、0 に近いほど関係なし、です。

## 2. 2つの遺伝子で、散布図をかいてみる
**散布図**は、1人を1つの点にして、横に遺伝子A、縦に遺伝子Bの値をとったグラフです。

まず、500個の中で **いちばん相関が強い2つの遺伝子** をさがします。

In [ ]:
corr = X_top.corr()                       # 500×500 の相関係数の表を作る
c = corr.where(~np.eye(len(corr), dtype=bool))  # 自分自身との相関(必ず1)は除く
a, b = c.stack().idxmax()                 # いちばん大きい組み合わせ
print(f"いちばん相関が強いのは {a} と {b}:相関係数 = {corr.loc[a, b]:.2f}")

plt.scatter(X_top[a], X_top[b], s=8)
plt.xlabel(a); plt.ylabel(b); plt.title(f"{a} と {b}(1つの点が1人)");

**読み方**:点が右上がりの直線にそってならんでいれば、2つの遺伝子は「いっしょに働く」ことが多い、ということです。

生物学では、いっしょに働く遺伝子は、同じスイッチ(遺伝子のはたらきを調節するしくみ)で動いていたり、同じ仕事を分担していたりすることがあります。

## 3. 相関がない2つも見てみよう

In [ ]:
a0, b0 = c.abs().stack().idxmin()        # 相関がいちばん 0 に近い組み合わせ
print(f"{a0} と {b0}:相関係数 = {corr.loc[a0, b0]:.2f}")
plt.scatter(X_top[a0], X_top[b0], s=8)
plt.xlabel(a0); plt.ylabel(b0); plt.title("相関がほぼない2つ");

## 4. たくさんの遺伝子の相関を、一度に見る(ヒートマップ)
**ヒートマップ**は、表の数字を色で表したものです。ここでは30個の遺伝子どうしの相関係数を色にします。
赤いほど正の相関、青いほど負の相関、白は相関なしです。

In [ ]:
g30 = X_top.columns[:30]
sns.clustermap(X_top[g30].corr(), cmap="RdBu_r", vmin=-1, vmax=1, figsize=(8, 8))
plt.suptitle("30個の遺伝子どうしの相関", y=1.02);

**読み方**:似た動きをする遺伝子が近くに並ぶように、自動で並べかえています(`clustermap` の機能)。
赤いかたまりは「いっしょに動く遺伝子のグループ」です。このように「似たものどうしをまとめる」考え方は、04 のクラスター分析につながります。

## 5. 大事な話:相関は「原因」を教えてくれない
有名な例:**アイスクリームがよく売れる日ほど、水の事故が多い**。
では、アイスを食べると事故が起きるのでしょうか? ちがいますね。
本当の理由は **暑さ** です。暑い日はアイスも売れるし、泳ぐ人も増えます。

このように、2つの両方に影響している「かくれた理由」を **交絡(こうらく)因子** といいます。
相関があっても、「AがBの原因」(**因果**)とは言えないのです。

### 遺伝子データでも起きるか、確かめてみよう
手順2の2つの遺伝子は、本当に「いっしょに働く仲間」なのでしょうか?
もしかすると、**がんの種類** がかくれた理由かもしれません。
同じ散布図を、がんの種類で色分けしてみます。

In [ ]:
sns.scatterplot(x=X_top[a], y=X_top[b], hue=y, s=12)
plt.title(f"{a} と {b}(がんの種類で色分け)");

次に、**がんの種類ごとに分けて** 相関係数を計算します。
全員まとめたときの相関と比べて、どうでしょう?

In [ ]:
print(f"全員まとめて:{corr.loc[a, b]:.2f}")
for k, idx in y.groupby(y).groups.items():
    r = X_top.loc[idx, a].corr(X_top.loc[idx, b])
    print(f"{k} だけ:{r:.2f}")

🤔 **考えてみよう**
- がんの種類ごとに分けても、相関は強いままですか? それとも弱くなりましたか?
- **弱くなった** 場合:強い相関は「がんの種類によって、2つとも高い/2つとも低い」ことから生まれていた、と考えられます。がんの種類が交絡因子になっていた、ということです。
- **強いまま** の場合:同じがんの中でもいっしょに動くので、2つの遺伝子が本当に連動している可能性が高まります。それでも「Aが原因でBが変わる」とまでは、このデータからは言えません。

原因を確かめるには、片方の遺伝子のはたらきを止めて、もう片方がどうなるかを見る **実験** が必要です。
データ分析で分かるのは「手がかり」まで、ということを覚えておきましょう。

## 今日のまとめ
- 相関は、2つの数字がいっしょに動く度合い。−1〜+1 の相関係数で表す
- 散布図とヒートマップで、相関を目で見られる
- 相関があっても因果とは限らない。かくれた理由(交絡因子)に注意する
- 遺伝子データでは、「がんの種類」や「組織のちがい」がかくれた理由になりやすい

**次回(03)**:2万個の数字を、2つの数字にまとめて「地図」をかく方法、**主成分分析** をやります。

### もっと知りたい人へ(公式の説明)
- pandas の相関係数 `corr`:https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html
- seaborn の `clustermap`:https://seaborn.pydata.org/generated/seaborn.clustermap.html
- 総務省統計局「なるほど統計学園」相関:https://www.stat.go.jp/naruhodo/